In [ ]:
import rasterio
from rasterio.sample import sample_gen

raster = rasterio.open("DK001_KØBENHAVN_UA2012_DHM_V030.tif")

# Example: list of (x, y) points in EPSG:3035
points = [(7310000, 1040000), (7320000, 1041000)]

for val in raster.sample(points):
    print(val[0])  # height in meters

In [ ]:
pip install pyproj

In [ ]:
from pyproj import Transformer
import rasterio

# Transform to EPSG:3035
transformer = Transformer.from_crs(4326, 3035, always_xy=True)
lon, lat = 12.5683, 55.6761  # Copenhagen center
x, y = transformer.transform(lon, lat)

# Sample
with rasterio.open("DK001_KØBENHAVN_UA2012_DHM_V030.tif") as src:
    val = list(src.sample([(x, y)]))[0][0]
    print(val)

In [ ]:
import math
from datetime import datetime

def solar_altitude_azimuth(year, month, day, hour, minute, latitude, longitude, timezone_offset):
    """
    Calculate solar altitude and azimuth angles.
    :param year: int
    :param month: int
    :param day: int
    :param hour: int (local clock time)
    :param minute: int
    :param latitude: float in degrees
    :param longitude: float in degrees
    :param timezone_offset: float (e.g. +2 for CEST)
    :return: (altitude_deg, azimuth_deg)
    """
    lat_rad = math.radians(latitude)

    # Day of the year
    date = datetime(year, month, day)
    n = date.timetuple().tm_yday

    # Fractional local time
    decimal_hour = hour + minute / 60.0

    # Declination δ
    decl = 23.45 * math.sin(math.radians(360/365 * (284 + n)))
    decl_rad = math.radians(decl)

    # Equation of time (in minutes)
    B = math.radians(360/365 * (n - 81))
    EoT = 9.87 * math.sin(2*B) - 7.53 * math.cos(B) - 1.5 * math.sin(B)

    # Local standard time meridian
    LSTM = 15 * timezone_offset

    # Time correction
    TC = 4 * (longitude - LSTM) + EoT

    # Local solar time
    LST = decimal_hour + TC/60.0

    # Hour angle
    H = 15 * (LST - 12)
    H_rad = math.radians(H)

    # Altitude angle
    sin_alpha = math.sin(lat_rad) * math.sin(decl_rad) + \
                math.cos(lat_rad) * math.cos(decl_rad) * math.cos(H_rad)
    alpha_rad = math.asin(sin_alpha)
    alpha_deg = math.degrees(alpha_rad)

    # Azimuth calculation
    cos_alpha = math.cos(alpha_rad)
    if cos_alpha == 0:
        azimuth_deg = float('nan')  # sun directly overhead
    else:
        sin_az = -math.cos(decl_rad) * math.sin(H_rad) / cos_alpha
        cos_az = (math.sin(decl_rad) - math.sin(lat_rad) * math.sin(alpha_rad)) / (math.cos(lat_rad) * cos_alpha)
        az_rad = math.atan2(sin_az, cos_az)
        azimuth_deg = math.degrees(az_rad)
        # Adjust to 0-360°
        azimuth_deg = (azimuth_deg + 360) % 360

    return alpha_deg, azimuth_deg


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import rasterio
from rasterio.windows import Window

def sun_check(year = 2026, 
        month = 4, 
        day = 17, 
        hour = 8, 
        minute = 00, 
        latitude = 55.40305, 
        longitude = 12.34105, 
        timezone_offset=2):
    # Parameters
    window_size = 50  # 50x50 pixels ~ 500x500 m
    row, col = 1518, 2694  # City Hall row/col

    with rasterio.open(path) as src:
        r0 = max(0, row - window_size//2)
        c0 = max(col - window_size//2, 0)
        subset = src.read(1, window=Window(c0, r0, window_size, window_size)).astype('float32')
        subset = np.where(subset == src.nodata, np.nan, subset)
        
        # Sample the center pixel (or small window)
        center_row = window_size // 2
        center_col = window_size // 2
        height = subset[center_row, center_col]
        if np.isnan(height):
            height_text = ""
        else:
            height_text = f"{height:.1f} m"

    # Plot
    plt.figure(figsize=(8,8))
    plt.imshow(subset, cmap='terrain', origin='upper')
    plt.colorbar(label='Building height (m)')

    # Plot the X
    plt.scatter(center_col, center_row, c='red', marker='x', s=100, label='City Hall')

    # Annotate height next to the X
    plt.text(center_col + 1, center_row - 1, height_text, color='black', fontsize=12, weight='bold')

    import math
    alpha_deg, azimuth_deg = solar_altitude_azimuth(
        year, month, day, 
        hour, minute, 
        latitude, longitude, 
        timezone_offset)

    # --- Sun direction vector in pixel space ---
    az = math.radians(azimuth_deg)

    # Raster coordinates:
    #   row increases downward  → north = negative row direction
    #   col increases rightward → east = positive col direction
    dcol =  np.sin(az)       # east-west component
    drow = -np.cos(az)       # north-south component

    # Length of the line in pixels
    line_length = 30

    # Endpoint of the line
    end_col = center_col + dcol * line_length
    end_row = center_row + drow * line_length

    # --- Sample heights along the sun direction ---
    num_samples = 40
    rows = center_row + drow * np.linspace(1, num_samples, num_samples)
    cols = center_col + dcol * np.linspace(1, num_samples, num_samples)

    sampled_heights = []
    for r, c in zip(rows, cols):
        r_int = int(round(r))
        c_int = int(round(c))
        if 0 <= r_int < subset.shape[0] and 0 <= c_int < subset.shape[1]:
            sampled_heights.append(subset[r_int, c_int])
        else:
            sampled_heights.append(np.nan)

    sampled_heights = np.array(sampled_heights)

    # --- Compute skyline elevation angles ---
    observer_height = 1.8
    relative_heights = sampled_heights - observer_height

    pixel_size = 10  # meters per pixel (adjust if needed)
    distances = np.linspace(1, num_samples, num_samples) * pixel_size/10

    elev_angles = np.degrees(np.arctan2(relative_heights, distances))
    max_skyline_angle = np.nanmax(elev_angles)

    # --- Determine if sun is blocked ---
    sun_alt = alpha_deg
    line_color = "red" if max_skyline_angle > sun_alt else "green"

    # --- Plot the line ---
    plt.plot(
        [center_col, end_col],
        [center_row, end_row],
        color=line_color,
        linewidth=2,
        label=f"Sun direction ({'blocked' if line_color=='red' else 'visible'})"
    )

    plt.title("Building Heights near Copenhagen City Hall")
    plt.legend()
    plt.show()

sun_check(year = 2026, 
        month = 4, 
        day = 17, 
        hour = 13, 
        minute = 00, 
        latitude = 55.40305, 
        longitude = 12.34105, 
        timezone_offset=2)

In [ ]:
import math

# Convert azimuth to radians
az = math.radians(azimuth_deg)

# Pixel direction (raster coordinates)
dcol =  np.sin(az)      # east-west
drow = -np.cos(az)      # north-south

line_length = 40  # pixels

end_col = center_col + dcol * line_length
end_row = center_row + drow * line_length

# Plot the line
plt.plot(
    [center_col, end_col],
    [center_row, end_row],
    color='yellow',
    linewidth=2,
    label='Sun direction'
)
